In [ ]:
from pydrake.all import (
    DiagramBuilder,
    LeafSystem,
    MultibodyPlant,
    MultibodyPositionToGeometryPose,
    Parser,
    SceneGraph,
    Simulator,
    StartMeshcat,
    StateInterpolatorWithDiscreteDerivative,
    ApplyMultibodyPlantConfig,
    ModelDirectives,
    ProcessModelDirectives,
    VisualizationConfig,
    ApplyVisualizationConfig,
    ApplyLcmBusConfig,
    LcmSubscriberSystem,
    Value,
    Context,
    BasicVector,
)
from manipulation.station import (
    LoadScenario,
)
from manipulation.utils import (
    RenderDiagram,
    ConfigureParser
)

import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
from lcmdefs.messages.so101 import lcmt_so101_configuration

import numpy as np
import math

In [ ]:
meshcat = StartMeshcat()

In [ ]:
class SO101StatusReceiver(LeafSystem):
    def __init__(self):
        super().__init__()

        self.input_port = self.DeclareAbstractInputPort(
            name="lcmt_observation",
            model_value=Value(lcmt_so101_configuration())
        )
        self.DeclareVectorOutputPort(
            name="position_measured",
            size=6,
            calc=self.ParseObservation
        )

    def ParseObservation(self, context: Context, output: BasicVector) -> None:
        observation: lcmt_so101_configuration = self.input_port.Eval(context)
        q_current = np.array(observation.q)
        output.SetFromVector(q_current)

In [ ]:
scenario = LoadScenario(filename="../scenarios/so101_hardware.yaml")
builder = DiagramBuilder()
scene_graph = SceneGraph()
builder.AddNamedSystem("scene_graph", scene_graph)
plant = MultibodyPlant(time_step=scenario.plant_config.time_step)
ApplyMultibodyPlantConfig(scenario.plant_config, plant)
plant.RegisterAsSourceForSceneGraph(scene_graph)
parser = Parser(plant)
ConfigureParser(parser)

added_models = ProcessModelDirectives(
    directives=ModelDirectives(directives=scenario.directives),
    parser=parser,
)

plant.Finalize()

to_pose = MultibodyPositionToGeometryPose(plant)
builder.AddSystem(to_pose)
builder.Connect(
    to_pose.get_output_port(),
    scene_graph.get_source_pose_port(plant.get_source_id()),
)

config = VisualizationConfig()
config.publish_contacts = False
config.publish_inertia = False
ApplyVisualizationConfig(
    config,
    builder=builder,
    plant=plant,
    scene_graph=scene_graph,
    meshcat=meshcat,
)

lcm_buses = ApplyLcmBusConfig(lcm_buses=scenario.lcm_buses, builder=builder)

lcm = lcm_buses.Find("SO101 Bus", "so101_lcm")

so101_status_receiver = SO101StatusReceiver()
builder.AddNamedSystem("so101.status_receiver", so101_status_receiver)
so101_status_subscriber = LcmSubscriberSystem.Make(
    channel="SO101_STATUS",
    lcm_type=lcmt_so101_configuration,
    lcm=lcm,
    use_cpp_serializer=False,
    wait_for_message_on_initialization_timeout=10,
)
builder.AddNamedSystem("so101.status_subscriber", so101_status_subscriber)

builder.ExportOutput(
    so101_status_receiver.get_output_port(),
    "so101.position_measured"
)

interpolator = StateInterpolatorWithDiscreteDerivative(6, 1e-4)
builder.AddNamedSystem("so101.state_interpolator", interpolator)
builder.Connect(
    so101_status_receiver.get_output_port(),
    interpolator.get_input_port()
)
builder.ExportOutput(
    interpolator.get_output_port(),
    "so101.state_estimated"
)

builder.Connect(
    so101_status_subscriber.get_output_port(),
    so101_status_receiver.get_input_port()
)

builder.Connect(
    so101_status_receiver.get_output_port(),
    to_pose.get_input_port()
)

diagram = builder.Build()
diagram.set_name("SO101StationInterface")

simulator = Simulator(diagram)
simulator.set_target_realtime_rate(1.0)
simulator.AdvanceTo(math.inf, interruptible=True)